# **Notebook 3 – Categorical Feature Engineering**

In [20]:
# Load dataset
import pandas as pd
df = pd.read_csv("telecom_customer_churn_dataset.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Customer_ID        2000 non-null   object 
 1   Gender             2000 non-null   object 
 2   Region             2000 non-null   object 
 3   Contract_Type      1980 non-null   object 
 4   Internet_Service   1980 non-null   object 
 5   Payment_Method     1970 non-null   object 
 6   Device_Protection  2000 non-null   object 
 7   Tech_Support       2000 non-null   object 
 8   Streaming_Service  2000 non-null   object 
 9   Customer_Segment   2000 non-null   object 
 10  Age                2000 non-null   int64  
 11  Tenure_Months      2000 non-null   int64  
 12  Monthly_Charges    2000 non-null   float64
 13  Total_Charges      2000 non-null   float64
 14  Support_Calls      2000 non-null   int64  
 15  Data_Usage_GB      2000 non-null   float64
 16  Joining_Date       2000 

## **1. Categorical Variables**

**Understand the Concept**

* Categorical variables contain groups or categories instead of continuous numbers.
* Examples are `Gender`, `Region`, and `Payment_Method`.
* These values usually need to be converted into numbers before using them in many ML models.



## **2. Nominal Variables**

**Understand the Concept**

* Nominal variables are categories with no specific order.
* One category is not higher or lower than another.
* Examples include `Gender`, `Region`, and `Payment_Method`.

**Demonstrate the Concept**

**Example:**

* `Payment_Method` has categories such as UPI, Credit Card, and Cash.
* There is no natural ranking between these categories.



In [21]:
# nominal variables - unique categories
print("Gender:", df["Gender"].dropna().unique())
print("Region:", df["Region"].dropna().unique())
print("Payment Method:", df["Payment_Method"].dropna().unique())

Gender: ['Male' 'Female']
Region: ['Other' 'East' 'Northeast' 'West' 'Central' 'South' 'North']
Payment Method: ['Net Banking' 'UPI' 'Auto Pay' 'Debit Card' 'Credit Card' 'Cash']


**Explanation**

* These columns contain different categories without any natural order.
* For example, UPI is not greater or smaller than Credit Card.

## **3. Ordinal Variables**

**Understand the Concept**

* Ordinal variables have categories with a meaningful order.
* The categories can be arranged from lower to higher.
* The difference between categories may not be equal.

**Demonstrate the Concept**

**Example:**

* `Customer_Segment` can be considered ordered as `Basic → Standard → Premium`.
* This order represents increasing customer value.



In [22]:
# Customer_Segment - categories
df["Customer_Segment"].value_counts()

Customer_Segment
Standard    518
Business    515
Basic       488
Premium     479
Name: count, dtype: int64

In [23]:
# customer segments - order

segment_order = {"Basic": 1,"Standard": 2,"Premium": 3,"Business": 4}

df["Customer_Segment_Ordinal"] = (df["Customer_Segment"].map(segment_order))
df[["Customer_Segment", "Customer_Segment_Ordinal"]].head()

,Customer_Segment,Customer_Segment_Ordinal
0,Premium,3
1,Business,4
2,Basic,1
3,Business,4
4,Premium,3


**Explanation**

* Each customer segment is assigned a number based on its order.
* `Basic` receives a lower value, while higher-value segments receive higher values.

**Justify Your Feature**

* I created this feature because the customer segments have a meaningful order.

## **4. Binary Variables**

**Understand the Concept**

* Binary variables have only two possible categories.
* Common examples are `Yes/No` and `Male/Female`.
* They can be easily converted into numerical values such as `0` and `1`.

**Demonstrate the Concept**

**Example:**

* `Tech_Support` contains `Yes` and `No`.
* We can represent `Yes` as `1` and `No` as `0`.



In [24]:
# binary feature - Tech_Support 
df["Tech_Support_Binary"] = (df["Tech_Support"].map({"No": 0, "Yes": 1}))
df[["Tech_Support", "Tech_Support_Binary"]].head()

,Tech_Support,Tech_Support_Binary
0,Yes,1
1,Yes,1
2,No,0
3,No,0
4,No,0


## **5. One-Hot Encoding**

**Understand the Concept**

* One-Hot Encoding converts categories into separate columns.
* Each category gets a value of `0` or `1`.
* It is mainly used for nominal variables with no natural order.

**Demonstrate the Concept**

**Example:**

* `Payment_Method` contains categories such as UPI, Credit Card, and Cash.
* One-Hot Encoding creates a separate column for each category.

**AI/ML Usage:**

* It converts categorical data into numerical features that ML models can use.

In [25]:
#  One-Hot Encoding - Payment_Method

payment_encoded = pd.get_dummies(df["Payment_Method"], prefix="Payment",dtype=int)
payment_encoded.head()

,Payment_Auto Pay,Payment_Cash,Payment_Credit Card,Payment_Debit Card,Payment_Net Banking,Payment_UPI
0,0,0,0,0,1,0
1,0,0,0,0,0,0
2,0,0,0,0,1,0
3,0,0,0,0,0,1
4,1,0,0,0,0,0


In [26]:
# Add the encoded col to dataset

df = pd.concat([df, payment_encoded], axis=1)

df[["Payment_Method"] + payment_encoded.columns.tolist()].head()

,Payment_Method,Payment_Auto Pay,Payment_Cash,Payment_Credit Card,Payment_Debit Card,Payment_Net Banking,Payment_UPI
0,Net Banking,0,0,0,0,1,0
1,NaN,0,0,0,0,0,0
2,Net Banking,0,0,0,0,1,0
3,UPI,0,0,0,0,0,1
4,Auto Pay,1,0,0,0,0,0


**Explanation**

* Each payment method is converted into a separate column.
* `1` means the customer used that payment method, while `0` means they did not.
* This is useful for nominal categorical variables.

## **6. Ordinal Encoding**

**Understand the Concept**

* Ordinal Encoding converts ordered categories into numbers.
* The numbers follow the natural order of the categories.
* It should be used only when the categories have a meaningful order.

**Demonstrate the Concept**

**Example:**

* `Customer_Segment` can be ordered as `Basic → Standard → Premium → Business`.

**AI/ML Usage:**

* It helps the model understand the order between categories.

In [27]:
from sklearn.preprocessing import OrdinalEncoder

segment_order = [["Basic", "Standard", "Premium", "Business"]]

encoder = OrdinalEncoder(categories=segment_order)

df["Customer_Segment_Encoded"] = encoder.fit_transform(df[["Customer_Segment"]])
df[["Customer_Segment", "Customer_Segment_Encoded"]].head()

,Customer_Segment,Customer_Segment_Encoded
0,Premium,2.0
1,Business,3.0
2,Basic,0.0
3,Business,3.0
4,Premium,2.0


**Explanation**

* Each customer segment is converted into a number based on its order.
* `Basic` gets the lowest value and `Business` gets the highest value.
* The original order of the categories is preserved.

**Justify Your Feature**

* I selected Ordinal Encoding because `Customer_Segment` has a meaningful order.

## **7. Label Encoding**

**Understand the Concept**

* Label Encoding changes each category into a number.
* Each category gets a unique number.
* It is commonly used for encoding the target variable.

**Demonstrate the Concept**

**Example:**

* The `Churn` target has two categories: `No` and `Yes`.
* We can convert them into `0` and `1`.

**AI/ML Usage:**

* Label Encoding makes categorical target values suitable for classification models.

In [28]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()

df["Churn_Encoded"] = label_encoder.fit_transform(df["Churn"])
df[["Churn", "Churn_Encoded"]].sample(3)

,Churn,Churn_Encoded
1439,No,0
1068,No,0
1509,Yes,1


**Explanation**

* `No` and `Yes` are converted into numerical labels.
* The encoded target can be used by a classification model.
* Label Encoding should be used carefully for input features because the numbers may imply an order.



## **8. Frequency Encoding**

**Understand the Concept**

* Frequency Encoding replaces each category with how often it appears.
* It converts categorical values into numerical values.
* It is useful when a categorical feature has many categories.

**Demonstrate the Concept**

**Example:**

* `Region` can be replaced by the number of customers belonging to each region.

**AI/ML Usage:**

* Frequency Encoding gives the model information about how common each category is.

In [29]:
# region frequency
region_frequency = df["Region"].value_counts()

# frequency-encoded feature
df["Region_Frequency"] = df["Region"].map(region_frequency)
df[["Region", "Region_Frequency"]].head()

,Region,Region_Frequency
0,Other,276
1,East,306
2,Northeast,287
3,West,255
4,Other,276


**Explanation**

* Each region is replaced with its number of occurrences.
* Common regions receive higher frequency values.
* The original categorical column is kept unchanged.

**Justify Your Feature**

* I created this feature because it shows how common each region is in the dataset.

## **9. Target Encoding**

**Understand the Concept**

* Target Encoding replaces a category with a statistic calculated from the target.
* For classification, the statistic is commonly the target mean.
* It can be useful for categorical features with many categories.
* It must be handled carefully because it can cause data leakage.

**Demonstrate the Concept**

**Example:**

* We can calculate the average churn rate for each `Internet_Service` category.

**AI/ML Usage:**

* Target Encoding provides the model with information about the relationship between a category and the target.

In [30]:
# Convert target to numerical values
df["Churn_Flag"] = df["Churn"].map({"No": 0,"Yes": 1})

# Calculate average churn rate for each internet service
target_mean = df.groupby("Internet_Service")["Churn_Flag"].mean()

# Create target-encoded feature
df["Internet_Service_Target_Encoded"] = (df["Internet_Service"].map(target_mean))
df[["Internet_Service","Internet_Service_Target_Encoded"]].head()

,Internet_Service,Internet_Service_Target_Encoded
0,DSL,0.310940
1,No Internet,0.332627
2,5G,0.347305
3,No Internet,0.332627
4,Fiber Optic,0.323045


**Explanation**

* The average churn rate is calculated for each internet service category.
* Each category is replaced with its corresponding churn rate.
* In real ML workflows, target encoding should be calculated using training data only to prevent leakage.

**Justify Your Feature**

* I selected this feature because it shows the relationship between internet service type and churn.

## **10. Rare Category Grouping**

**Understand the Concept**

* Rare categories are categories that appear very few times.
* We can combine them into one category such as `Other`.
* This helps simplify features with many uncommon categories.

**Demonstrate the Concept**

**Example:**

* If a region has very few customers, we can group it into `Other`.

**AI/ML Usage:**

* Rare category grouping reduces unnecessary categories and can help prevent overfitting.

In [31]:
# Calculate region frequencies
region_counts = df["Region"].value_counts()

# Set minimum frequency
min_frequency = 100

# Group rare regions
df["Region_Grouped"] = df["Region"].where(df["Region"].map(region_counts) >= min_frequency,"Other")
df[["Region", "Region_Grouped"]].head()

,Region,Region_Grouped
0,Other,Other
1,East,East
2,Northeast,Northeast
3,West,West
4,Other,Other


**Explanation**

* Regions with fewer than 100 records are grouped into `Other`.
* Common regions remain unchanged.
* This makes the categorical feature simpler.

**Justify Your Feature**

* I grouped rare categories because very small groups may provide limited reliable information.

## **11. Category Cardinality**

**Understand the Concept**

* Cardinality means the number of unique categories in a categorical feature.
* Low-cardinality features have fewer categories.
* High-cardinality features have many categories.
* High cardinality can make encoding more difficult.

**Demonstrate the Concept**

**Example:**

* `Gender` has low cardinality, while a feature containing thousands of different locations would have high cardinality.

**AI/ML Usage:**

* Cardinality helps us choose a suitable encoding technique.

In [32]:
# Check cardinality of categorical columns

categorical_columns = df.select_dtypes(include="object").columns
cardinality = df[categorical_columns].nunique()
cardinality.sort_values(ascending=False)

Customer_ID          2000
Joining_Date         1448
Region                  7
Region_Grouped          7
Payment_Method          6
Internet_Service        4
Customer_Segment        4
Contract_Type           3
Gender                  2
Device_Protection       2
Tech_Support            2
Streaming_Service       2
Churn                   2
dtype: int64

**Explanation**

* `nunique()` counts the number of unique categories.
* The result helps identify low- and high-cardinality features.
* Encoding methods should be selected based on the number of categories.

**Justify Your Feature**

* I checked cardinality because the number of categories helps decide which encoding method to use.

## **12. Handling Unknown Categories**

**Understand the Concept**

* An unknown category is a value that was not present during model training.
* New categories can appear when the model receives new data.
* The encoding method should handle these values safely.
* This prevents errors when processing new data.

**Demonstrate the Concept**

**Example:**

* Suppose the training data contains `South`, `North`, and `West`, but new data contains `Northeast`.
* The encoder should handle the new category instead of producing an error.

**AI/ML Usage:**

* Handling unknown categories makes the preprocessing process more reliable for new data.

In [33]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore",sparse_output=False)
# Fit using training-like data
encoded_region = encoder.fit_transform(df[["Region"]])
encoded_region[:5]

array([[0., 0., 0., 0., 1., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1., 0., 0.]])

In [34]:
# New data containing an unseen category

new_data = pd.DataFrame({"Region": ["South", "Northeast", "West"]})
new_encoded_region = encoder.transform(new_data)
new_encoded_region

array([[0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1.]])

**Explanation**

* `handle_unknown="ignore"` prevents an error when a new category appears.
* The unseen category is represented using zeros across the known category columns.
* This is useful when the model receives new data after training.

**Justify Your Feature**

* I used unknown-category handling because new customer categories may appear in future data.

## **13. Combining Categories**

**Understand the Concept**

* Combining categories means grouping related categories into one category.
* It helps reduce the number of categories.
* It can make a feature simpler and easier for the model to use.

**Demonstrate the Concept**

**Example:**

* `Payment_Method` has several categories such as `UPI`, `Credit Card`, `Debit Card`, and `Net Banking`.
* We can combine card-based payments into one category called `Card`.

**AI/ML Usage:**

* Combining categories can reduce complexity and create more meaningful groups.

In [35]:
# Combine related payment methods

payment_group = {"Credit Card": "Card","Debit Card": "Card","UPI": "Digital","Net Banking": "Digital","Auto Pay": "Digital","Cash": "Cash"}
df["Payment_Group"] = df["Payment_Method"].map(payment_group)
df[["Payment_Method", "Payment_Group"]].head()

,Payment_Method,Payment_Group
0,Net Banking,Digital
1,NaN,NaN
2,Net Banking,Digital
3,UPI,Digital
4,Auto Pay,Digital


**Explanation**

* Related payment methods are grouped into broader categories.
* `Credit Card` and `Debit Card` become `Card`.
* Digital payment methods are grouped into `Digital`.

**Justify Your Feature**

* I combined related payment methods because broader groups can simplify the categorical feature.